# Model 5 · BiLSTM + Attention (from scratch)

No pre-trained weights: word embedding table, BiLSTM, and attention pooling
are all learned from this corpus alone. Independent of Models 1-4.


In [ ]:
import os, re, gc, math, random, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Config - CPU only
# ---------------------------------------------------------------------------
BASE        = "../data"
TRAIN_PATH  = f"{BASE}/train.csv"
TEST_PATH   = f"{BASE}/test.csv"
OUTPUT_PATH = "../outputs/submission.csv"

OPTIONS  = ["A", "B", "C", "D", "E"]
SEED     = 42
VAL_SIZE = 0.20
N_FOLDS  = 5

MPNET_ID = "sentence-transformers/all-mpnet-base-v2"

# ---------------------------------------------------------------------------
# Weights & Biases - one run per model, so every model has a tracked run with
# comparable metrics (MAP@3, accuracy, macro F1, weighted F1).
# Wrapped so that a W&B failure can never abort the run or lose the submission.
# ---------------------------------------------------------------------------
WANDB_PROJECT = "23f2004742-t22026"
WANDB_ON = True

os.environ["WANDB_SILENT"] = "true"

try:
    import wandb
    _key = None
    try:
        from kaggle_secrets import UserSecretsClient
        _key = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        _key = os.environ.get("WANDB_API_KEY")

    if _key:
        wandb.login(key=_key)
        print(f"W&B ready -> project '{WANDB_PROJECT}'")
    else:
        # A bare wandb.login() waits on stdin, which would hang a
        # "Save & Run All" commit forever. Disable instead of risking that.
        WANDB_ON = False
        print("W&B key not found (add WANDB_API_KEY as a Kaggle secret).")
        print("Continuing without tracking; all metrics are still printed.")
except Exception as e:
    WANDB_ON = False
    print(f"W&B unavailable ({type(e).__name__}); metrics still printed locally")

# DeBERTa is kept as an experiment only. It is a 435M model: fine-tuning it on
# CPU is not practical, so it stays off unless a GPU is attached.
RUN_DEBERTA = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
print(f"torch {torch.__version__} | device: {DEVICE}")
print(f"DeBERTa experiment: {'ON' if RUN_DEBERTA else 'OFF (no GPU - expected on CPU)'}")


## 1. Load data

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train : {train_df.shape}")
print(f"Test  : {test_df.shape}")

counts = train_df["answer"].value_counts().reindex(OPTIONS)
probs  = counts / counts.sum()
order  = list(probs.sort_values(ascending=False).index)

PRIOR_MAP3 = probs[order[0]] + probs[order[1]] / 2 + probs[order[2]] / 3
LABEL_LOGPRIOR = np.log(probs[OPTIONS].values.astype(np.float64))

print("\nAnswer distribution:")
for o in order:
    print(f"  {o}  {counts[o]:>4}  ({probs[o]:.1%})")
print(f"\nRandom-ordering MAP@3        : 0.3667")
print(f"Always '{' '.join(order[:3])}' MAP@3          : {PRIOR_MAP3:.4f}   <- the bar to beat")

train_df.head(3)


## 2. Preprocessing and split

Prompts carry boilerplate prefixes ("Pick the best possible answer:", …) which
are stripped. **The split is made on the CLEANED prompt**, because two prompts
that differ only by prefix become identical after cleaning - splitting on raw
text would put the same question on both sides and make validation meaningless.


In [ ]:
BOILERPLATE = [
    r"^Pick the best possible answer:\s*", r"^Choose the correct answer:\s*",
    r"^Select the most accurate option:\s*", r"^Identify the correct statement:\s*",
    r"^Determine the correct option:\s*", r"^Which of the following\s*",
    r"\s*among the listed options\.?$", r"\s*from the following choices\.?$",
    r"\s*based on the given context\.?$", r"\s*carefully\.?$",
]


def clean_text(t):
    t = re.sub(r"\s+", " ", str(t)).strip()
    for p in BOILERPLATE:
        t = re.sub(p, "", t, flags=re.IGNORECASE).strip()
    return t


def clean_frame(df):
    out = df.copy()
    for c in ["prompt"] + OPTIONS:
        out[c] = out[c].map(clean_text)
    return out


train = clean_frame(train_df)
test  = clean_frame(test_df)

n_raw, n_clean = train_df["prompt"].nunique(), train["prompt"].nunique()
print(f"Unique prompts  raw {n_raw}  ->  cleaned {n_clean}   "
      f"({n_raw - n_clean} collapsed by preprocessing)")

uniq = train["prompt"].unique()
tr_p, va_p = train_test_split(uniq, test_size=VAL_SIZE, random_state=SEED)

train_split = train[train["prompt"].isin(tr_p)].reset_index(drop=True)
val_split   = train[train["prompt"].isin(va_p)].reset_index(drop=True)
assert not (set(train_split["prompt"]) & set(val_split["prompt"]))

y_val   = val_split["answer"].tolist()
y_train = train["answer"].tolist()

print(f"Train {len(train_split)} | Val {len(val_split)} | no cleaned-prompt overlap")


## 3. Metrics - MAP@3, Accuracy, Macro F1

In [ ]:
def average_precision_at_3(actual, predicted):
    for rank, p in enumerate(predicted[:3], start=1):
        if p == actual:
            return 1.0 / rank
    return 0.0


def map_at_3(actuals, preds):
    return float(np.mean([average_precision_at_3(a, p) for a, p in zip(actuals, preds)]))


def top3(scores):
    """(n,5) score matrix -> list of top-3 letter lists."""
    return [[OPTIONS[i] for i in np.argsort(-s)][:3] for s in np.asarray(scores)]


def wandb_run(name, metrics, config=None):
    """One W&B run per model. Never allowed to break the pipeline."""
    if not WANDB_ON:
        return
    try:
        wandb.init(project=WANDB_PROJECT, name=name, reinit=True,
                   config=config or {})
        wandb.log({k.lower().replace("@", "").replace("+", "_"): float(v)
                   for k, v in metrics.items()})
        wandb.finish()
    except Exception as e:
        print(f"    (W&B log skipped: {type(e).__name__})")


def evaluate(name, actuals, scores, store=None, log=True, config=None):
    preds = top3(scores)
    t1 = [p[0] for p in preds]
    m = {
        "MAP@3":    map_at_3(actuals, preds),
        "Accuracy": accuracy_score(actuals, t1),
        "MacroF1":  f1_score(actuals, t1, labels=OPTIONS, average="macro", zero_division=0),
        "WgtF1":    f1_score(actuals, t1, labels=OPTIONS, average="weighted", zero_division=0),
    }
    print(f"{name:<28} MAP@3 {m['MAP@3']:.4f} | Acc {m['Accuracy']:.4f} | "
          f"MacroF1 {m['MacroF1']:.4f} | WgtF1 {m['WgtF1']:.4f}")
    if store is not None:
        store[name] = m
    if log:
        wandb_run(name.strip().replace(" ", "-").lower(), m, config)
    return m


RESULTS = {}
print(f"reference: random 0.3667 | prior {PRIOR_MAP3:.4f}")


In [ ]:
# Row positions of the validation split within the cleaned training frame.
# (Hoisted from the source notebook's Model 2 cell -- every model below needs it.)
val_pos = train.index[train["prompt"].isin(va_p)].to_numpy()


## Model 5 - BiLSTM + Attention  *(model from scratch)*

No pre-trained weights anywhere: the word embedding table is learned from this
corpus alone.

```
  "question <sep> option"
          |
   Embedding(V, 128)          <- learned from scratch
          |
   BiLSTM(128 -> 2x128)
          |
   attention:  a = softmax(w.h) ,  pooled = sum(a * h)
          |
   Linear(256 -> 64) -> ReLU -> Linear(64 -> 1)
          |
   softmax over the 5 options -> cross-entropy
```

Attention pooling lets the model weight the tokens that decide the answer instead
of averaging the whole sequence.


In [ ]:
from collections import Counter

MAX_LEN, MIN_FREQ, EMB_SZ, HID = 96, 2, 128, 128
# Attention over a learned embedding table needs enough passes to converge;
# cutting this short was leaving the model undertrained.
BILSTM_EPOCHS = 30


def toks(s):
    return re.findall(r"[a-z0-9']+", str(s).lower())


cnt = Counter()
for _, r in train.iterrows():
    cnt.update(toks(r["prompt"]))
    for o in OPTIONS:
        cnt.update(toks(r[o]))

VOCAB = {"<pad>": 0, "<unk>": 1, "<sep>": 2}
for w, c in cnt.most_common():
    if c >= MIN_FREQ:
        VOCAB[w] = len(VOCAB)
print(f"vocabulary: {len(VOCAB)} types")


def encode_pair(p, o):
    ids = ([VOCAB.get(w, 1) for w in toks(p)][:MAX_LEN // 2] + [2] +
           [VOCAB.get(w, 1) for w in toks(o)])[:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))


def seq_ids(df):
    return torch.tensor(np.array(
        [[encode_pair(r["prompt"], r[o]) for o in OPTIONS] for _, r in df.iterrows()],
        dtype=np.int64))


IDS_train, IDS_test = seq_ids(train), seq_ids(test)


class BiLSTMAttn(nn.Module):
    def __init__(self, vocab, emb=EMB_SZ, hid=HID, p=0.3):
        super().__init__()
        self.emb  = nn.Embedding(vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(2 * hid, 1)
        self.drop = nn.Dropout(p)
        self.head = nn.Sequential(nn.Linear(2 * hid, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, ids):                        # (B,5,L)
        B, K, L = ids.shape
        flat = ids.reshape(B * K, L)
        mask = flat != 0
        h, _ = self.lstm(self.emb(flat))
        a = self.attn(h).squeeze(-1).masked_fill(~mask, -1e9)
        pooled = self.drop((h * torch.softmax(a, -1).unsqueeze(-1)).sum(1))
        return self.head(pooled).reshape(B, K)


def fit_bilstm(idx, yy, seed):
    torch.manual_seed(seed)
    ids = IDS_train[idx].to(DEVICE)
    y = torch.tensor(yy, dtype=torch.long, device=DEVICE)
    model = BiLSTMAttn(len(VOCAB)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=BILSTM_EPOCHS)
    for ep in range(BILSTM_EPOCHS):
        model.train()
        perm = torch.randperm(len(y))
        for k in range(0, len(y), 32):
            b = perm[k:k + 32]
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(ids[b]), y[b])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    return model


@torch.no_grad()
def bilstm_predict(model, idx, which, bs=64):
    model.eval()
    ids = IDS_train[idx] if which == "train" else IDS_test
    out = [model(ids[k:k + bs].to(DEVICE)).cpu().numpy() for k in range(0, len(ids), bs)]
    return np.concatenate(out).astype(np.float32)


print(f"Training BiLSTM+Attention ({BILSTM_EPOCHS} epochs/fold), 5-fold...")
bl_oof, bl_test = run_folds(fit_bilstm, bilstm_predict, len(train), len(test),
                            "bilstm", n_seeds=2)

bl_val = bl_oof[val_pos]
evaluate("5. BiLSTM + Attention", y_val, bl_val, RESULTS)
evaluate("   BiLSTM OOF (2000)", y_train, bl_oof)
